In [2]:
# ================================
# TRAINING CODE
# ================================

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# 1. LOAD DATA
df = pd.read_csv("clean.csv")

# 2. CLEAN DATA
df = df.drop(columns=["Unnamed: 0", "Date", "Pond_ID"], errors="ignore")
df = df.drop(columns=["Risk_Level"], errors="ignore")

df = pd.get_dummies(df, drop_first=True)
df = df.fillna(df.mean())

# 3. FEATURES & TARGETS
X = df.drop(columns=["Risk_Score", "Feed_Efficiency"])
y_risk = df["Risk_Score"]
y_feed = df["Feed_Efficiency"]

# 🔥 SAVE COLUMNS (IMPORTANT)
joblib.dump(X.columns, "columns.pkl")

# 4. SPLIT
X_train, X_test, y_risk_train, y_risk_test = train_test_split(
    X, y_risk, test_size=0.2, random_state=42
)

_, _, y_feed_train, y_feed_test = train_test_split(
    X, y_feed, test_size=0.2, random_state=42
)

# 5. SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# 6. MODELS
risk_model = RandomForestRegressor(n_estimators=100, random_state=42)
feed_model = RandomForestRegressor(n_estimators=100, random_state=42)

risk_model.fit(X_train_scaled, y_risk_train)
feed_model.fit(X_train_scaled, y_feed_train)

# 7. SAVE EVERYTHING
joblib.dump(risk_model, "risk_model.pkl")
joblib.dump(feed_model, "feed_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("✅ Training complete & models saved!")

✅ Training complete & models saved!


In [ ]:
import pandas as pd
import joblib

# Load models
risk_model = joblib.load("risk_model.pkl")
feed_model = joblib.load("feed_model.pkl")
scaler = joblib.load("scaler.pkl")
columns = joblib.load("columns.pkl")

# ================================
# USER INPUT (ONLY IMPORTANT ONES)
# ================================
input_data = {
    "Water_Temp_C": float(input("Water Temperature (°C): ")),
    "Rainfall_mm": float(input("Rainfall (mm): ")),
    "pH_Level": float(input("pH Level: ")),
    "Ammonia_ppm": float(input("Ammonia (ppm): ")),
    "Dissolved_Oxygen_mgL": float(input("Dissolved Oxygen (mg/L): ")),
    "Fish_Count": float(input("Fish Count: ")),
    "Daily_Feed_kg": float(input("Daily Feed (kg): ")),
    "Est_Avg_Weight_g": float(input("Avg Fish Weight (g): "))
}

# Categorical inputs
season = input("Season (Summer/Winter/Monsoon): ")
species = input("Species (Rohu / Rohu-Bhakur Polyculture): ")

# ================================
# CREATE FULL INPUT
# ================================
input_df = pd.DataFrame([input_data])

# Fill all missing columns with 0
for col in columns:
    if col not in input_df.columns:
        input_df[col] = 0

# Handle categorical columns
if f"Season_{season}" in input_df.columns:
    input_df[f"Season_{season}"] = 1

if f"Target_Species_{species}" in input_df.columns:
    input_df[f"Target_Species_{species}"] = 1

# Final column order
input_df = input_df[columns]

# ================================
# SCALE & PREDICT
# ================================
input_scaled = scaler.transform(input_df)

risk_pred = risk_model.predict(input_scaled)
feed_pred = feed_model.predict(input_scaled)

# ================================
# OUTPUT
# ================================
print("\n===== RESULT =====")
print(f"Predicted Risk Score: {risk_pred[0]:.4f}")
print(f"Predicted Feed Efficiency: {feed_pred[0]:.4f}")


===== RESULT =====
Predicted Risk Score: 1.5000
Predicted Feed Efficiency: 0.0396
